In [ ]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import numpy as np
from cop_ard_app.computation import create_client
from cop_ard_app.dem import import_dem, stack_dem
from cop_ard_app.io import Path, normpath, read, glob, delete
from cop_ard_app.area_projection import stack_geometry, stack_mask
from cop_ard_app.simulation import stack_simulation, stack_simulation_ml
from cop_ard_app.slc import import_slc_stack, finalize_import, extract_orbits

from cop_ard_app.model import update_coreg, extract_acquisition_dates
from cop_ard_app.utils import get_bbox
from cop_ard_app.etad import import_ETAD
from cop_ard_app.orbit import import_ORBIT
from cop_ard_app.coreg import stack_coregistration
from cop_ard_core.logger import logger
from cop_ard_app.utils import get_bbox

In [ ]:
in_files = glob("/data/SENTINEL1/SURAT/SAFE/*SAFE")
path = "/data/SENTINEL1/SURAT/NOTEBOOK"
wdir_path_static = f"{path}/STATIC"
wdir_path_tmp = f"{path}/DYNAMIC_SECONDARY"
slcs_path = f"{wdir_path_tmp}/slcs"
dem_path = f"{path}/DYNAMIC/dem"

indices = [0, 1, 27, 28]
orbit_files = None
if orbit_files is not None:
    orbit_files = [Path(e) for e in orbit_files]
etad_files = glob(f"/data/SENTINEL1/SURAT/ETAD/*ETA*")
if etad_files is not None:
    etad_files = [Path(e) for e in etad_files]
chunks_sr = {"acquisition_date": 1, "polarization": 1, "azimuth_time": 2048, "slant_range_time": 2048}
chunks = {"x": 2048, "y": 2048}
chunk_length = max([v for v in chunks.values()])
spacing = {"x": 30, "y": 30}
spacing_gslc = {"x": 50, "y": 50}
calib_type = "BetaZero"
noise_removal_flag = False
crs = None
recompute = False
multiburst = True
do_import = False if (recompute and multiburst) else True

In [ ]:
type, n_jobs, job_memory = ["LOCAL", 16, 0]
client = create_client(type, n_jobs, job_memory)
client

In [ ]:
delete(slcs_path, recursive=True, allow_non_existent=True)
slcs = import_slc_stack(
    in_files,
    slcs_path,
    deramp=True,
    multiburst=multiburst,
    noise_removal_flag=noise_removal_flag,
    apply_invalid_mask_flag=True,
    indices=indices,
    calib_type=calib_type,
    chunks=chunks_sr,
    do_import=do_import,
)


In [ ]:
slcs = xr.DataTree()
if multiburst:
    files = glob(f"{slcs_path}/bursts/*/orig/*")
    for f in files:
        name = f.removeprefix(slcs_path)
        slcs[name] = read(f)
else:
    slcs = read(slcs_path)

In [ ]:
bbox = get_bbox(slcs)
multiburst = True if "bursts" in slcs else False

In [ ]:
used_etad = []
used_orbits = []
acquisition_dates = extract_acquisition_dates(slcs, multiburst=multiburst)
if orbit_files is not None:
    slcs, used_orbits = import_ORBIT(
        slcs,
        acquisition_dates,
        orbit_files,
        multiburst=multiburst,
        wdir_path=wdir_path_tmp,
    )
if etad_files is not None:
    slcs, used_etad = import_ETAD(
        slcs,
        acquisition_dates,
        etad_files,
        multiburst=multiburst,
        wdir_path=wdir_path_tmp,
        enabled=False,
    )

In [ ]:
params = {
    "dem": "...",
    "atbd_link": "Algorithm Theoretical Basis Document (ATBD) for Sentinel-1 Analysis Ready Data (ARD) Processing, Version 1.0. Published online. Available at: TBD",
    "facility": "ESA-Copernicus Ground Segment",
    "product_timeliness": "P24D",
    "product_timeliness_category": "NTC",
    "product_data_access": "https://link_product",
    "product_pixel_coordinate_convention": "pixel-center",
    "cslc_status": "nominal",
    "gslc_status": "nominal",
    "nrb_status": "nominal",
    "measurement_type": calib_type,
    "product_filtering": "TBD",
    "eastern_geometric_accuracy": {"bias": np.nan, "stddev": np.nan},
    "northern_geometric_accuracy": {"bias": np.nan, "stddev": np.nan},
    "geometric_accuracy_radial_rmse": np.nan,
    "product_family_specification": "TBD",
    "noise_removal": noise_removal_flag,
    "providers": [
        {"name": "ESA", "description": "European Space Agency", "roles": ["producer"]},
        {"name": "Sentinel-1 ARD Core", "description": "Analysis Ready Data Processor", "roles": ["processor"]},
    ],
}
if used_etad:
    params["etad"] = used_etad
if used_orbits:
    params["orbit"] = used_orbits

In [ ]:
slcs, reference_date = finalize_import(
    slcs=slcs,
    working_directory=wdir_path_static,
    recompute=recompute,
)

In [ ]:
dem = import_dem(
    bbox=bbox,
    dem_type=dem_path,
    working_dir=wdir_path_tmp,
    chunks=chunks,
)

In [ ]:
dem_tree = stack_dem(
    dem,
    slcs,
    multiburst,
    spacing,
    wdir_path_static,
    wdir_path_tmp,
    chunks,
)

In [ ]:
orbits = extract_orbits(
    slcs,
    multiburst=multiburst,
    acquisition_date=[reference_date],
)

In [ ]:
simulation_tree = stack_simulation(
    dem_tree=dem_tree,
    orbits=orbits,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    chunks={**chunks, "acquisition_date": 1},
    master_image=True,
    recompute=recompute,
)

In [ ]:
simulation_ml_tree = stack_simulation_ml(
    simulation_tree=simulation_tree,
    reference_date=reference_date,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    recompute=recompute,
)

In [ ]:
stack_geometry(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    working_directory_static=wdir_path_static,
    working_directory_tmp=wdir_path_tmp,
    reference_date=reference_date,
    chunk_length=chunk_length,
    recompute=recompute,
    multiburst=multiburst,
)

In [ ]:
stack_mask(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    working_directory_static=wdir_path_static,
    reference_date=reference_date,
    recompute=recompute,
    multiburst=multiburst,
)

In [ ]:
orbits = extract_orbits(
    slcs,
    multiburst=multiburst,
)
orbits = orbits.where(orbits.acquisition_date != reference_date, drop=True)

In [ ]:
simulation_secondary_tree = stack_simulation(
    dem_tree=dem_tree,
    orbits=orbits,
    multiburst=multiburst,
    working_directory=wdir_path_tmp,
    chunks={**chunks, "acquisition_date": 1},
    master_image=False,
    recompute=recompute,
)

In [ ]:
# Checkpoint: Restart execution from here if the program crashes.

simulation_secondary_tree = xr.DataTree()
for k in simulation_tree.bursts if multiburst else {"data": simulation_tree}:
    path = f"bursts/{k}" if multiburst else "data"
    data_path = normpath(f"{wdir_path_tmp}/{path}/simulation")
    if multiburst:
        simulation_secondary_tree[path] = read(data_path)
    else:
        simulation_secondary_tree = read(data_path)

In [ ]:
slcs = stack_coregistration(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    reference_date,
    simulation_secondary_tree=simulation_secondary_tree,
    multiburst=multiburst,
    working_directory=wdir_path_tmp,
    enable_dask=True,
    recompute=True,
)